In [1]:
from google.colab import drive
drive.mount("/content/gdrive/", force_remount=True)
import pandas as pd
import numpy as np
import os
import openpyxl
import time
from datetime import timedelta

Mounted at /content/gdrive/


In [2]:
def UserNeighbor(user,k, active_user , sim_path):
  #Đọc file độ tương đồng của user
    user_df = pd.read_excel(r"{0}/User{1}.xlsx".format(sim_path, user))

    #Lấy user theo top k sim cao
    list_user_SS = []

    for i in user_df.User.to_list():
        if i not in active_user:
            list_user_SS.append(i)
        if len(list_user_SS) == k:
            break

    # Sắp xếp lại danh sách theo độ tương đồng từ cao đến thấp
    list_user_SS = sorted(list_user_SS, key=lambda x: user_df[user_df['User'] == x]['Sim'].values[0], reverse=True)

    return list_user_SS


In [3]:
def ItemNeighbor(item, sim_path, k):
    # Đọc file độ tương đồng của item
    item_df = pd.read_excel(r"{0}/Item{1}.xlsx".format(sim_path, item))

    # Lấy danh sách item láng giềng của item
    list_item_neighbor = item_df['Item'][:k].tolist()

    return list_item_neighbor

In [4]:
#Kiểm tra nếu hơn 50% láng giềng không đánh giá thì loại
def check(user_list, item, training_df):
  existing_users_count = sum((training_df['User'].isin(user_list)) & (training_df['Item'] == item))
  if existing_users_count < 0.5 * len(user_list):
      return False
  else:
      return True

In [5]:
#Input: user, i:item, level: mức độ level đệ quy hiện tại, k: số lượng láng giềng
#Output: rating dự đoán của user trên item i

def RecursivePrediction(user, item, training_df, user_average_rating_df, level, sim_path, k, active_user, level_rated):
    level_rated = level + 2
    if level >= 2:
        return -1, level_rated

    active_user.append(user)

    neighbor_list = UserNeighbor(user, k, active_user, sim_path)
    # Gọi hàm kiểm tra số lượng láng giềng
    if check(neighbor_list, item, training_df) == False:
      level_rated = 4
      return -1, level_rated
    
    user_similarity_df = pd.read_excel(r"{}/User{}.xlsx".format(sim_path, user))

    alpha = 0.0
    beta = 0.0

    for neighbor in neighbor_list:
        #Kiểm tra nếu láng giềng có đánh giá trên item
        if not training_df[(training_df['User'] == neighbor) & (training_df['Item'] == item)].empty:
            rating = training_df.loc[(training_df['User'] == neighbor) & (training_df['Item'] == item), 'Rating'].values[0]
            avg_rating = user_average_rating_df.loc[user_average_rating_df['User'] == neighbor, 'UserAvg'].values[0]
            sim = user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0]
            alpha += ((int(rating *10 ** decimal_places) /10 ** decimal_places) - avg_rating) * sim
            beta += abs(sim)
        #Nếu láng giềng không đánh giá thì gọi đệ quy dự đoán
        else:
            rating_j, level_rated = RecursivePrediction(neighbor, item, training_df, user_average_rating_df, level + 1, sim_path, k, active_user, level_rated)
            active_user.pop(-1)

            if rating_j == -1:
                return -1, level_rated
            #Thực hiện phép tính sau khi thoát đệ quy
            avg_rating = user_average_rating_df.loc[user_average_rating_df['User'] == neighbor, 'UserAvg'].values[0]
            sim = user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0]
            alpha += 0.5 * ((int(rating_j  *10 ** decimal_places) /10 ** decimal_places) - avg_rating) * sim
            beta += 0.5 * abs(sim)

    if beta == 0:
        return user_average_rating_df.loc[user_average_rating_df['User'] == user, 'UserAvg'].values[0], level_rated
    return user_average_rating_df.loc[user_average_rating_df['User'] == user, 'UserAvg'].values[0] + (alpha / beta), level_rated


In [6]:
def Prediction(user, item, training_df, user_average_rating_df, level, sim_path, active_user, k, sim_item_path):
    active_user = [user]
    neighbors_positions_col = []
    level_counts = [0, 0, 0, 0, 0, 0]

    #Lấy danh sách láng giềng
    neighbor_list = UserNeighbor(user, k, active_user, sim_path)
    user_similarity_df = pd.read_excel(r"{}/User{}.xlsx".format(sim_path, user))
    user_in_similarity = user_similarity_df['User'].tolist()

    neighbor_positions = user_similarity_df[user_similarity_df['User'].isin(neighbor_list)].index.tolist()

    alpha = 0.0
    beta = 0.0
    check_real_user = 0

    for neighbor in neighbor_list:
        check_real_user += 1
        rating_neighbor = 0.0
        level_rated = 0
        #Kiểm tra nếu láng giềng có sim bằng 0
        if user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0] == 0:
            break
        
        #Kiểm tra nếu láng giềng có đánh giá trên item
        if not training_df[(training_df['User'] == neighbor) & (training_df['Item'] == item)].empty:
            rating = training_df.loc[(training_df['User'] == neighbor) & (training_df['Item'] == item), 'Rating'].values[0]
            avg_rating = user_average_rating_df.loc[user_average_rating_df['User'] == neighbor, 'UserAvg'].values[0]
            sim = user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0]
            alpha += ((int(rating  *10 ** decimal_places) /10 ** decimal_places) - avg_rating) * sim
            beta += abs(sim)
        else:
            item_sim_path = r"{}/Item{}.xlsx".format(sim_item_path, item)
            #Kiểm tra có tồn tại file Sim Item
            if os.path.exists(item_sim_path):
                item_similarity_df = pd.read_excel(item_sim_path)
                neighbor_item_list = ItemNeighbor(item, sim_item_path, k + 10)
                check_neighbor_rate = 0

                for neighbor_item in neighbor_item_list:
                    if not training_df[(training_df['User'] == neighbor) & (training_df['Item'] == neighbor_item)].empty:
                        #Thực hiện tính toán
                        rating = training_df.loc[(training_df['User'] == neighbor) & (training_df['Item'] == neighbor_item), 'Rating'].values[0]
                        avg_rating = user_average_rating_df.loc[user_average_rating_df['User'] == neighbor, 'UserAvg'].values[0]
                        user_sim = user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0]
                        item_sim = item_similarity_df.loc[item_similarity_df['Item'] == neighbor_item, 'Sim'].values[0]
                        alpha += ((int(rating  *10 ** decimal_places) /10 ** decimal_places) - avg_rating) * user_sim * item_sim
                        beta += abs(user_sim * item_sim)
                        check_neighbor_rate = 1
                        level_rated = 1
                        break
                #Thực hiện đệ quy nếu chưa tìm được rating cho item
                if check_neighbor_rate == 0:
                    rating_neighbor, level_rated = RecursivePrediction(neighbor, item, training_df, user_average_rating_df, 0, sim_path, k, active_user, level_rated)

                    if (check_real_user <= k and level_rated == 4):
                      level_counts[level_rated] += 1

                    #Nếu không dự đoán được rating thì chuyển qua neighbor kế
                    if rating_neighbor == -1:
                        if len(user_in_similarity) == len(neighbor_list):
                            break
                        new_neighbor = user_in_similarity[len(neighbor_list)]
                        neighbor_positions = [pos for pos in neighbor_positions if user_similarity_df.loc[pos, 'User'] != neighbor]
                        neighbor_list.append(new_neighbor)
                        neighbor_positions.extend(user_similarity_df[user_similarity_df['User'].isin([new_neighbor])].index.tolist())
                        level_rated = 5
                        level_counts[level_rated] += 1
                        continue
                    #Thực hiện phép tính sau khi thoát đệ quy
                    avg_rating = user_average_rating_df.loc[user_average_rating_df['User'] == neighbor, 'UserAvg'].values[0]
                    sim = user_similarity_df.loc[user_similarity_df['User'] == neighbor, 'Sim'].values[0]
                    alpha += 0.5 * ((int(rating_neighbor  *10 ** decimal_places) /10 ** decimal_places) - avg_rating) * sim
                    beta += 0.5 * abs(sim)
            else:
                return -1, neighbors_positions_col, level_counts
        level_counts[level_rated] += 1
    neighbors_positions_col.append(neighbor_positions)
    if beta == 0:
        return user_average_rating_df.UserAvg[(user_average_rating_df.User == user)].values[0], neighbors_positions_col, level_counts
    return user_average_rating_df.UserAvg[user_average_rating_df.User == user].values[0] + (alpha / beta), neighbors_positions_col, level_counts

In [7]:
def recommend_list(user, N, k, training_df, testing_df, user_average_rating_df, real_pre_df, sim_path, sim_item_path):
    user_col = []
    user_col_rmse = []
    item_col = []
    rating_real_col = []
    rating_predict_col = []
    neighbors_positions_col_list = []
    level_rated_col = []
    count_have_rate_col=[]
    count_item_rate_col=[]
    count_level0_col=[]
    count_level1_col=[]
    count_level2_col=[]
    count_append_col=[]

    levels = [4, 3, 2, 1, 0]
    switcher_values = [2, 1, 0, -1, -2]

    switcher = {
      -2: "Have rate",
      -1: "Item rate",
      0: "Level 0 rate",
      1: "Level 1 rate",
      2: "Level 2 rate"
    }

    predict_list = testing_df.Item[testing_df.User == user].tolist()
    predict_list.sort()

    for j in predict_list:
        ripa_rating, neighbors_positions, level_counts = Prediction(user, j, training_df, user_average_rating_df, 0, sim_path, [user], k, sim_item_path)
        if ripa_rating == -1:
            continue

        rate_real = testing_df.Rating[(testing_df.User == user) & (testing_df.Item == j)].values[0]
        user_col.append(user)
        item_col.append(j)
        rating_real_col.append(rate_real)
        rating_predict_col.append(ripa_rating)

        for level, switch_value in zip(levels, switcher_values):
          if level_counts[level] > 0:
              level_rated_col.append(switcher.get(switch_value))
              break

        count_have_rate_col.append(level_counts[0])                         # count user have rate
        count_item_rate_col.append(level_counts[1])                         # count user have rate neighbor item
        count_level0_col.append(level_counts[2])                            # count user have rate level 0
        count_level1_col.append(level_counts[3])                            # count user have rate level 1
        count_level2_col.append(level_counts[4])                            # count user have rate level 2
        count_append_col.append(level_counts[5])

        neighbors_positions_col_list.append(neighbors_positions)
        user_col_rmse.append(np.sqrt(pow(ripa_rating - rate_real,2)))

    df = pd.DataFrame(data={
        'User': user_col,
        'Item': item_col,
        'Rating_real': rating_real_col,
        'Rating_predict': rating_predict_col,
        'Neighbors_Positions': neighbors_positions_col_list,
        'Level': level_rated_col,
        'Have rate': count_have_rate_col,
        'Item rate': count_item_rate_col,
        'Level 0': count_level0_col,
        'Level 1': count_level1_col,
        'Level 2': count_level2_col,
        'Append user': count_append_col,
        'RMSE': user_col_rmse,
    })
    df = df.sort_values(by=['Rating_predict'], axis=0, ascending=False, ignore_index=True)
    real_pre_df = pd.concat([real_pre_df, df])

    df = df.iloc[:N]

    recommended_items = df.Item.tolist()
    neighbors_positions_col = []

    for positions in neighbors_positions_col_list[:N]:
        filtered_positions = [pos for sublist in positions for pos in sublist if pos not in set(df.index)]
        neighbors_positions_col.append(filtered_positions)

    return recommended_items, real_pre_df


In [8]:
def revaluation(N, k, sim_path, testing_df, training_df, first, last, sim_item_path):
    real_pre_df = pd.DataFrame()
    User_col = []
    I_N_col = []
    I_te_col = []
    intersection_col = []
    I_N_len_col = []
    I_te_len_col = []
    intersection_len_col = []
    Prec_u_N = []
    Rec_u_N = []
    F1_u_N = []

    testing_user = testing_df.User[testing_df.User >= 0].to_list()
    testing_user = list(dict.fromkeys(testing_user))

    for i in range(first, last + 1):
        if i in testing_user and i in training_df.User.to_list():
            I_N, real_pre_df = recommend_list(i, N, k, training_df, testing_df, user_avg_df, real_pre_df, sim_path, sim_item_path)
            I_te = testing_df.Item[testing_df.User == i][testing_df.Rating > 3].to_list()
            intersection = [value for value in I_N if value in I_te]
            try:
                Prec = abs(len(intersection)) / abs(len(I_N))
                Rec = abs(len(intersection)) / abs(len(I_te))
                F1 = 2 * (Prec * Rec) / (Prec + Rec)
            except ZeroDivisionError:
                Prec = 0
                Rec = 0
                F1 = 0

            User_col.append(i)
            I_N_col.append(' '.join(str(e) for e in I_N))
            I_te_col.append(' '.join(str(e) for e in I_te))
            intersection_col.append(' '.join(str(e) for e in intersection))
            I_N_len_col.append(len(I_N))
            I_te_len_col.append(len(I_te))
            intersection_len_col.append(len(intersection))
            Prec_u_N.append(Prec)
            Rec_u_N.append(Rec)
            F1_u_N.append(F1)

    df = pd.DataFrame({
        'User': User_col,
        'Tập tư vấn (C)': I_N_col,
        'Tập đúng (T)': I_te_col,
        'Giao TC': intersection_col,
        'C_length': I_N_len_col,
        'T_length': I_te_len_col,
        'TC_length': intersection_len_col,
        'Precision': Prec_u_N,
        'Recall': Rec_u_N,
        'F1': F1_u_N
    })

    if df['C_length'].sum() > 0:
        pre = df['TC_length'].sum() / df['C_length'].sum()
    else:
        pre = 0

    if df['T_length'].sum() > 0:
        recall = df['TC_length'].sum() / df['T_length'].sum()
    else:
        recall = 0

    if pre + recall > 0:
        f1 = 2 * pre * recall / (pre + recall)
    else:
        f1 = 0

    df_all = pd.DataFrame({'Precision_system': [pre], 'Recall_system': [recall], 'F1_system': [f1]})

    all_rmse = np.sqrt(real_pre_df.RMSE.sum() / real_pre_df.RMSE.notnull().sum())
    real_pre_df_all = pd.DataFrame({
        'RMSE_system': [all_rmse]
    })

    result_path = os.path.join(sim_path[:-7], 'Result_Idea_m_position_detail')
    if not os.path.exists(result_path):
        os.makedirs(result_path)

    file_path = os.path.join(result_path, "{}_N{}_k{}_idea_m_position_detail.xlsx".format(name, N, k))
    with pd.ExcelWriter(file_path) as writer:
        # Ghi từng DataFrame vào các cột riêng trong file Excel
        df.to_excel(writer, sheet_name = 'F1_RMSE', index = False) #F1
        df_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 10, index = False) #F1 tổng
        real_pre_df.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 13, index = False) #RMSE
        real_pre_df_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 26, index = False) #RMSE tổng


In [9]:
def convert_seconds_to_time(seconds):
    # Chuyển số giây thành đối tượng timedelta
    duration = timedelta(seconds=seconds)

    # Tính toán các thành phần ngày, giờ, phút và giây
    days = duration.days
    hours, remainder = divmod(duration.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    # Tạo chuỗi định dạng ngày:giờ:phút:giây
    formatted_time = "{:02}d:{:02}h:{:02}m:{:02}s".format(days, hours, minutes, seconds)

    return formatted_time

In [ ]:
# name = 'dlmau'
name = 'MusicInCar'
# name = 'Yelpitem'
# name = 'MusicInCar'
#name = 'MovieLen100k'
decimal_places = 9


#đọc file training, testing
training_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/{0}_training.xlsx'.format(name), dtype={'User': int, 'Item': int})
testing_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/{0}_testing.xlsx'.format(name), dtype={'User': int, 'Item': int})
#Đọc file User_Average
user_avg_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/User_Average.xlsx'.format(name))
user_avg_df = user_avg_df.rename(columns={'User_Average': 'UserAvg'}, errors="raise")

#số lượng user
n_user = np.max(training_df.User)
N = 10
k = 20
start_time = time.time()
revaluation(N, k,"/content/gdrive/MyDrive/Luận văn/{0}/SimUser".format(name), testing_df, training_df, 1, n_user, "/content/gdrive/MyDrive/Luận văn/{0}/SimItem".format(name))
end_time = time.time()

#đọc file kết quả
data = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/Result_Idea_m_position_detail/{0}_N{1}_k{2}_idea_m_position_detail.xlsx'.format(name, N, k))

#tạo dữ liệu cho time
time_all = pd.DataFrame({'Time_all': [convert_seconds_to_time(end_time - start_time)]})

#Thêm cột time vào file excel
file_path = os.path.join(r'/content/gdrive/MyDrive/Luận văn/{0}/Result_Idea_m_position_detail'.format(name), "{0}_N{1}_k{2}_idea_m_position_detail.xlsx".format(name, N, k))
with pd.ExcelWriter(file_path) as writer:
  data.to_excel(writer, sheet_name = 'F1_RMSE', index = False)
  time_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 27, index = False)

